In [1]:
import os
import sys
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from pathlib import Path
from cycler import cycler
import pickle

import pandas as pd 
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import statsmodels.api as sm

from scipy.stats import chi2
from scipy.stats import bartlett, ttest_ind

In [2]:
mpl.rcParams['axes.prop_cycle'] = cycler(color=[to_hex(i) for i in [
    (0.00, 0.45, 0.70),  # 1. Deep Blue (High contrast dark anchor)
    (0.90, 0.60, 0.00),  # 2. Orange (Distinct from yellow by saturation)
    (0.35, 0.70, 0.90),  # 3. Sky Blue (Lighter blue)
    (0.00, 0.60, 0.50),  # 4. Teal / Bluish Green (Heavy blue mix, safe for red-green CVD)
    (0.95, 0.90, 0.25),  # 5. Bright Yellow (Maximum luminance anchor)
    (0.80, 0.40, 0.70),  # 6. Pink / Magenta (Acts as a safe substitute for red)
    (0.20, 0.13, 0.53),  # 7. Indigo / Dark Purple (Distinct from Deep Blue by hue)
    (0.87, 0.80, 0.47),  # 8. Sand / Pale Gold (Muted yellow-brown)
    (0.27, 0.67, 0.60),  # 9. Mint / Seafoam (Lighter, desaturated teal)
    (0.65, 0.65, 0.65)   # 10. Medium Grey (Neutral baseline)
]])

# 1. Dataset Preparation

## 1.1. Load Data

In [3]:
%load_ext autoreload
%autoreload
import chip_utilities as utils

sys.path.insert(0, '..')
import sigmoid_fitting as sp
from experiment_data_loader import ExperimentDataLoader

exp_folder = "/Users/kautsarg/Documents/Final Project/Run Data/trial test data"
# exp_paths = [Path(exp_folder, name) for name in os.listdir(exp_folder) if name != ".DS_Store"]
exp_path = Path(exp_folder, "D20250808_E00_C00_F4500KHz_U_Sample_7")
curve_path = Path(exp_path, "preprocessed_curves_data.pkl")
save_path = os.path.join(exp_path, "curve_for_training.pkl")

with open(curve_path, 'rb') as f:
    processed_curve_results = pickle.load(f)

## 1.2. Build Dataset Combination

In [4]:
Y_well = processed_curve_results["well_labels"]
timestamps = processed_curve_results["timestamps"]

dataset_name = ["ori_curves", "ori_curves_avg"]
dataset = [
    processed_curve_results["curves"]["ori_curves"],
    processed_curve_results["curves"]["ori_curves_avg"]
]

for k, v in processed_curve_results["sigmoid_curves"].items():
    dataset_name.append(f"{k}_fitted_full")
    dataset.append(v["fitted_full"])
    dataset_name.append(f"{k}_fitted_stretched")
    dataset.append(v["fitted_stretched"])

dataset_name = np.array(dataset_name)
dataset = np.array(dataset)

## 1.3. Feature Extraction

In [5]:
%load_ext autoreload
%autoreload
import chip_utilities as utils

sys.path.insert(0, '..')
import sigmoid_fitting as sp
from experiment_data_loader import ExperimentDataLoader

def _process_single_row(y, X):
    valid = np.isfinite(X) & np.isfinite(y)
    if np.sum(valid) < 3:
        return {}
        
    try:
        return sp.extract_kinetic_parameters_original(X, y)
    except Exception:
        return {}

def extract_kinetic_features(timestamps, curves, n_jobs=-1):
    X = timestamps
    
    y_values_array = curves
    
    features = Parallel(n_jobs=n_jobs)(
        delayed(_process_single_row)(y, X) for y in y_values_array
    )
    
    return pd.DataFrame(features)

kinetics_path = os.path.join(exp_path, "initial_kinetics.pkl")
if (os.path.exists(kinetics_path)):
    with open(kinetics_path, 'rb') as f:
        kinetic_features = pickle.load(f)
else:
    kinetic_features = [extract_kinetic_features(timestamps, curves) for curves in dataset]
    
    with open(kinetics_path, 'wb') as f:
        pickle.dump(kinetic_features, f)
    print(f"  -> Saved initial kinetics features to {kinetics_path}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/Users/kautsarg/Documents/Final Project/POC_DDM/gk_code/0_4_AMCA Code on Chip/../sigmoid_fitting.py:16: RuntimeWarning: overflow encountered in power
  return Fm / (1.0 + np.exp(-Sc * (x - Cs)))**As + Fb
/Users/kautsarg/Documents/Final Project/POC_DDM/gk_code/0_4_AMCA Code on Chip/../sigmoid_fitting.py:27: RuntimeWarning: overflow encountered in power
  denominator = (1.0 + exp_term)**(As + 1)
/Users/kautsarg/Documents/Final Project/POC_DDM/gk_code/0_4_AMCA Code on Chip/../sigmoid_fitting.py:16: RuntimeWarning: overflow encountered in power
  return Fm / (1.0 + np.exp(-Sc * (x - Cs)))**As + Fb
/Users/kautsarg/Documents/Final Project/POC_DDM/gk_code/0_4_AMCA Code on Chip/../sigmoid_fitting.py:27: RuntimeWarning: overflow encountered in power
  denominator = (1.0 + exp_term)**(As + 1)
/Users/kautsarg/Documents/Final Project/POC_DDM/gk_code/0_4_AMCA Code on Chip/../sigmoid_fitting.py:16: RuntimeWarning: overflow encountered in power
  return Fm / (1.0 + np.exp(-Sc * (x - Cs)))**As + Fb
/U

  -> Saved initial kinetics features to /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/initial_kinetics.pkl


## 1.4. Shape Check

In [6]:
timestamps
Y_well
dataset_name
dataset
kinetic_features

print("timestamps", timestamps.shape)
print("Y_well", Y_well.shape)
for i, d in enumerate(dataset):
    print(dataset_name[i], "||", d.shape, "||", kinetic_features[i].shape)

timestamps (615,)
Y_well (12047,)
ori_curves || (12047, 615) || (12047, 42)
ori_curves_avg || (12047, 615) || (12047, 42)
original_fitted_full || (12047, 615) || (12047, 42)
original_fitted_stretched || (12047, 615) || (12047, 42)
cleaned_std_fitted_full || (12047, 615) || (12047, 42)
cleaned_std_fitted_stretched || (12047, 615) || (12047, 42)
cleaned_lowest_fitted_full || (12047, 615) || (12047, 42)
cleaned_lowest_fitted_stretched || (12047, 615) || (12047, 42)
avg_fitted_full || (12047, 615) || (12047, 42)
avg_fitted_stretched || (12047, 615) || (12047, 42)
avg_cleaned_std_fitted_full || (12047, 615) || (12047, 42)
avg_cleaned_std_fitted_stretched || (12047, 615) || (12047, 42)
avg_cleaned_lowest_fitted_full || (12047, 615) || (12047, 42)
avg_cleaned_lowest_fitted_stretched || (12047, 615) || (12047, 42)


# 2. MSC Outlier Detection

## 2.1. Features Check

In [7]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def feature_boxplot(features_df, well_labels, feature_columns, target="well", title="", save_path=None):
    plot_data = features_df.copy()
    plot_data[target] = well_labels
    valid_features = [f for f in feature_columns if f in plot_data.columns]
    
    if not valid_features:
        print(f"Skipping {title}: None of the requested features exist in this dataset.")
        return

    n_features = len(valid_features)
    
    if n_features == 3:
        fig = plt.figure(figsize=(10, 8))
        
        # 1. Add the 3 Boxplot axes
        ax1 = fig.add_subplot(2, 2, 1)
        ax2 = fig.add_subplot(2, 2, 2)
        ax3 = fig.add_subplot(2, 2, 3)
        axes = [ax1, ax2, ax3]
        
        for i, feature in enumerate(valid_features):
            sns.boxplot(data=plot_data, x=target, y=feature, ax=axes[i])
            axes[i].set_title(feature, fontweight='bold')
            axes[i].grid(True, alpha=0.3, axis='y')
            
        # 2. Add the 3D Scatter Plot axis
        ax4 = fig.add_subplot(2, 2, 4, projection='3d')
        f1, f2, f3 = valid_features
        
        # Plot each well separately so they get colored and added to the legend
        unique_targets = np.unique(well_labels)
        palette = sns.color_palette("tab10", len(unique_targets))
        
        for idx, val in enumerate(unique_targets):
            subset = plot_data[plot_data[target] == val]
            ax4.scatter(subset[f1], subset[f2], subset[f3], 
                        label=f"Well {val}", color=palette[idx], alpha=0.7, s=20)
            
        ax4.set_xlabel(f1, fontweight='bold')
        ax4.set_ylabel(f2, fontweight='bold')
        ax4.set_zlabel(f3, fontweight='bold')
        ax4.set_title("3D Feature Space", fontweight='bold')
        
        # Move legend slightly outside the 3D plot to avoid overlapping the data
        ax4.legend(title=target, bbox_to_anchor=(1.15, 1), loc='upper left')

    else:
        fig, axes = plt.subplots(1, n_features, figsize=(n_features * 4, 3))
        
        if n_features == 1:
            axes = [axes]

        for i, feature in enumerate(valid_features):
            sns.boxplot(data=plot_data, x=target, y=feature, ax=axes[i])
            axes[i].set_title(feature, fontweight='bold')
            axes[i].grid(True, alpha=0.3, axis='y')
            
    fig.suptitle(title, fontweight='bold', fontsize=14, y=1.02)
    plt.tight_layout()
    
    # --- CHANGED: Save and Close instead of Show ---
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300, transparent=False, facecolor='white')
        print(f"  -> Saved: {save_path}")
        
    plt.close(fig) # Closes the figure to free up memory and prevent it from rendering in notebooks


# ==========================================
# EXECUTION
# ==========================================

if (not os.path.exists(save_path)):
    msc_features = ["Ct", "Cy0", "log_F0"]
    msc_plot_path = f"{exp_path}/msc_outlier/"

    # Ensure the target directory exists
    os.makedirs(msc_plot_path, exist_ok=True)

    for name, features_df in zip(dataset_name, kinetic_features):
        
        clean_title = name.replace("_", " ").title()
        print(f"Generating plot for: {clean_title}...")
        
        # Construct the full file path for this specific dataset
        file_name = f"{name}_msc_features.png"
        save_file_path = os.path.join(msc_plot_path, file_name)
        
        feature_boxplot(
            features_df=features_df, 
            well_labels=Y_well, 
            feature_columns=msc_features, 
            title=f"Kinetic Features: {clean_title}",
            save_path=save_file_path
        )

Generating plot for: Ori Curves...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/msc_outlier/ori_curves_msc_features.png
Generating plot for: Ori Curves Avg...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/msc_outlier/ori_curves_avg_msc_features.png
Generating plot for: Original Fitted Full...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/msc_outlier/original_fitted_full_msc_features.png
Generating plot for: Original Fitted Stretched...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/msc_outlier/original_fitted_stretched_msc_features.png
Generating plot for: Cleaned Std Fitted Full...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/msc_outlier/cleaned

## 2.2. Line Fitting

In [8]:
def unsupervised_line_fitting(features):
    if isinstance(features, pd.DataFrame):
        X = features.values
    else:
        X = np.asarray(features)
    
    nan_mask = np.any(np.isnan(X), axis=1)
    X_clean = X[~nan_mask]
    n_samples = X_clean.shape[0]
    
    # Center the data
    mean_X = X_clean.mean(axis=0)
    X_centered = X_clean - mean_X
    
    # SVD
    U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
    
    # The principal line direction
    direction_ls = Vt[0, :]
    
    # Projections and Residuals
    projections_ls = X_centered @ direction_ls
    X_line_ls = projections_ls[:, np.newaxis] @ direction_ls[np.newaxis, :]
    X_line_ls_original = X_line_ls + mean_X
    residuals_ls = np.linalg.norm(X_clean - X_line_ls_original, axis=1)
    
    result = {
        'method': 'Least Squares (SVD)',
        'mean': mean_X,
        'direction': direction_ls,
        'all_directions': Vt,
        'singular_values': S,
        'n_samples': n_samples,
        'projections': projections_ls,
        'line_points': X_line_ls_original,
        'residuals': residuals_ls,
        'mse': np.mean(residuals_ls**2),
        'rmse': np.sqrt(np.mean(residuals_ls**2)),
    }
    return result

def print_line_equation(result, feature_names=["Ct", "Cy0", "log_F0"]):
    p0 = result['mean']
    v = result['direction']
    
    print("Parametric Line Equation:")
    for i, name in enumerate(feature_names):
        sign = "+" if v[i] >= 0 else "-"
        print(f"{name}(t) = {p0[i]:.4f} {sign} {abs(v[i]):.4f} * t")

def mahalanobis_distance_to_line(points, result):
    points = np.atleast_2d(points)
    
    # 1. Center the point(s) using the line's mean
    points_centered = points - result['mean']
    
    # 2. Extract the orthogonal directions (PC2 and PC3)
    # Vt[0] is the line. Vt[1:] are the perpendicular directions.
    orthogonal_directions = result['all_directions'][1:, :]
    
    # 3. Calculate variances for these directions
    # Variance = (Singular Value^2) / (N - 1)
    S_orthogonal = result['singular_values'][1:]
    variances = (S_orthogonal ** 2) / (result['n_samples'] - 1)
    
    # 4. Project points onto the orthogonal directions
    # Shape: (N_points, 2)
    projections = points_centered @ orthogonal_directions.T
    
    # 5. Calculate Mahalanobis distance
    # Sum of squared orthogonal projections divided by their variance
    mahalanobis_sq = np.sum((projections ** 2) / variances, axis=1)
    
    return np.sqrt(mahalanobis_sq)

def calculate_msc_mahalanobis(points, q1, q2, cov_matrix):
    points = np.atleast_2d(points)
    q1 = np.asarray(q1)
    q2 = np.asarray(q2)
    
    # The vector defining the line
    dq = q2 - q1
    
    numerator = np.dot(points - q1, dq)
    denominator = np.dot(dq, dq)
    P = numerator / denominator 
    
    p_proj = q1 + np.outer(P, dq)
    
    residual = points - p_proj
    
    # FIX: Use pseudo-inverse (pinv) to prevent Singular Matrix crashes
    inv_cov = np.linalg.pinv(cov_matrix)
    
    left_term = np.dot(residual, inv_cov)
    d_squared = np.sum(left_term * residual, axis=1)
    
    return np.sqrt(np.clip(d_squared, 0, None))


def calculate_chi2_threshold(p_value, df=2):
    chi2_val = chi2.ppf(1 - p_value, df)
    distance_threshold = np.sqrt(chi2_val)
    
    return distance_threshold

In [9]:
import numpy as np
import matplotlib.pyplot as plt

# ====================================================================
# MODULE 1: OUTLIER DETECTION (MATH & LOGIC)
# ====================================================================

def detect_outliers(features_df, Y_well, msc_features, p_value=0.001):
    """
    Fits 3D lines to feature spaces and calculates Mahalanobis distance to flag outliers.
    """
    msc_threshold = calculate_chi2_threshold(p_value=p_value, df=2) 
    unique_wells = np.unique(Y_well)
    
    features_df["msc_mahal_dist"] = np.nan
    features_df[f"msc_label_{p_value}"] = np.nan
    
    line_fittings_dict = {}

    for well in unique_wells:
        well_mask = (Y_well == well)
        valid_mask = well_mask & ~features_df[msc_features].isna().any(axis=1)
        
        features_clean = features_df.loc[valid_mask, msc_features].values
        
        if len(features_clean) > 0:
            well_line_fit = unsupervised_line_fitting(features_clean)
        else:
            well_line_fit = None
            
        line_fittings_dict[well] = well_line_fit
        
        if len(features_clean) == 0 or well_line_fit is None or len(well_line_fit.get('projections', [])) == 0:
            continue
            
        q1 = well_line_fit['mean']
        q2 = well_line_fit['mean'] + well_line_fit['direction']
        cov_matrix = np.cov(features_clean, rowvar=False)
        
        distances = calculate_msc_mahalanobis(features_clean, q1, q2, cov_matrix)
        
        features_df.loc[valid_mask, "msc_mahal_dist"] = distances
        features_df.loc[valid_mask, f"msc_label_{p_value}"] = np.array([-1 if d > msc_threshold else 1 for d in distances])
        
    return features_df, line_fittings_dict


# ====================================================================
# MODULE 2: VISUALIZATION (UPDATED TO 3 COLUMNS)
# ====================================================================
import os
import numpy as np
import matplotlib.pyplot as plt

def plot_outlier_results(features_df, curves_2d, reference_curves_2d, Y_well, msc_features, line_fittings_dict, dataset_name, p_value=0.001, save_path=None):
    """
    Generates a multi-row figure showing:
    Col 1: Reference curves (dataset[0]) highlighting the dataset[i] outliers.
    Col 2: Processed curves (dataset[i]) highlighting the dataset[i] outliers.
    Col 3: 3D Feature Space.
    """
    unique_wells = np.unique(Y_well)
    num_rows = min(len(unique_wells), 10)
    
    # Increased width to 24 to comfortably fit 3 columns
    fig = plt.figure(figsize=(24, num_rows * 4.5))
    
    for i, well in enumerate(unique_wells):
        if i >= num_rows:
            break
            
        well_mask = (Y_well == well)
        well_df = features_df[well_mask]
        
        # Boolean mask for outliers detected in the CURRENT dataset
        is_outlier = (well_df[f'msc_label_{p_value}'] == -1).fillna(False).values
        
        # Extract curves for both the Reference (dataset[0]) and Current (dataset[i])
        well_ref_curves = reference_curves_2d[well_mask]
        well_curr_curves = curves_2d[well_mask]
        
        normal_ref = well_ref_curves[~is_outlier]
        outlier_ref = well_ref_curves[is_outlier]
        
        normal_curr = well_curr_curves[~is_outlier]
        outlier_curr = well_curr_curves[is_outlier]
        
        # --- COLUMN 1: REFERENCE 2D CHART (dataset[0]) ---
        ax1 = fig.add_subplot(num_rows, 3, 3 * i + 1)
        
        if len(normal_ref) > 0:
            ax1.plot(normal_ref.T, c=f"C{i}", alpha=0.3)
        if len(outlier_ref) > 0:
            ax1.plot(outlier_ref.T, c="red", alpha=0.5, linewidth=1.5)
        if len(well_ref_curves) > 0:
            ax1.plot(np.nanmean(well_ref_curves, axis=0), c="black", linewidth=2.5)
            
        ax1.set_title(f"Well {well} | Original Amplification\n(n_outliers={len(outlier_ref)})", fontweight='bold')
        ax1.set_ylabel("Fluorescence")
        if i == num_rows - 1:
            ax1.set_xlabel("Time/Cycle")


        # --- COLUMN 2: CURRENT 2D CHART (dataset[i]) ---
        ax2 = fig.add_subplot(num_rows, 3, 3 * i + 2)
        
        if len(normal_curr) > 0:
            ax2.plot(normal_curr.T, c=f"C{i}", alpha=0.3)
        if len(outlier_curr) > 0:
            ax2.plot(outlier_curr.T, c="red", alpha=0.5, linewidth=1.5)
        if len(well_curr_curves) > 0:
            ax2.plot(np.nanmean(well_curr_curves, axis=0), c="black", linewidth=2.5)
            
        ax2.set_title(f"Well {well} | {dataset_name} Amplification\n(n_outliers={len(outlier_curr)})", fontweight='bold')
        if i == num_rows - 1:
            ax2.set_xlabel("Time/Cycle")


        # --- COLUMN 3: 3D SCATTER & FITTED LINE ---
        ax3 = fig.add_subplot(num_rows, 3, 3 * i + 3, projection='3d')
        
        X_val = well_df[msc_features[0]].values
        Y_val = well_df[msc_features[1]].values
        Z_val = well_df[msc_features[2]].values
        
        well_line_fit = line_fittings_dict.get(well)
        
        if well_line_fit is not None and len(well_line_fit.get('projections', [])) > 0:
            projections = well_line_fit['projections']
            p0 = well_line_fit['mean']
            v = well_line_fit['direction']
            
            t_min, t_max = np.min(projections), np.max(projections)
            margin = (t_max - t_min) * 0.1
            line_start = p0 + (t_min - margin) * v
            line_end = p0 + (t_max + margin) * v
            
            ax3.plot([line_start[0], line_end[0]], 
                      [line_start[1], line_end[1]], 
                      [line_start[2], line_end[2]], 
                      color='black', linewidth=2.5, label="Fitted Line")
            
        ax3.scatter(X_val[~is_outlier], Y_val[~is_outlier], Z_val[~is_outlier], 
                     c=f"C{i}", s=25, alpha=0.6, label="Normal")
        ax3.scatter(X_val[is_outlier], Y_val[is_outlier], Z_val[is_outlier], 
                     c="red", s=50, marker='x', alpha=1.0, label="Outlier")
        
        ax3.set_title(f"Well {well} | 3D Feature Space", fontweight='bold')
        ax3.set_xlabel(msc_features[0])
        ax3.set_ylabel(msc_features[1])
        ax3.set_zlabel(msc_features[2])
        
        if i == 0:
            ax3.legend(loc='upper left', bbox_to_anchor=(1.05, 1))

    fig.suptitle(f"Outlier Detection: {dataset_name} (p={p_value})", fontsize=18, fontweight='bold', y=1.01)
    plt.tight_layout()
    
    # --- CHANGED: Save and Close instead of Show ---
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300, facecolor='white')
        print(f"  -> Saved plot to: {save_path}")
        
    plt.close(fig)

In [10]:
# ====================================================================
# MODULE 3: MAIN EXECUTION LOOP
# ====================================================================

p_value = 0.001
all_line_fittings = []

# Define the reference dataset once (dataset[0])
reference_dataset_curves = dataset[0]
msc_plot_path = f"{exp_path}/msc_outlier/"

# Ensure the target directory exists
os.makedirs(msc_plot_path, exist_ok=True)

if (not os.path.exists(save_path)):
    for name, features_df, curves_2d in zip(dataset_name, kinetic_features, dataset):
        
        clean_title = name.replace("_", " ").title()
        print(f"\n{'='*60}\nProcessing Outliers for: {clean_title}\n{'='*60}")
        
        # 1. Detect Outliers (Updates features_df in place)
        features_df, line_fittings_dict = detect_outliers(
            features_df=features_df, 
            Y_well=Y_well, 
            msc_features=msc_features, 
            p_value=p_value
        )
        
        all_line_fittings.append(line_fittings_dict)
        
        # Construct the full file path for this specific dataset
        file_name = f"{name}_msc_outlier_results.png"
        save_file_path = os.path.join(msc_plot_path, file_name)
        
        # 2. Plot Results with Reference Dataset
        plot_outlier_results(
            features_df=features_df, 
            curves_2d=curves_2d, 
            reference_curves_2d=reference_dataset_curves,
            Y_well=Y_well, 
            msc_features=msc_features, 
            line_fittings_dict=line_fittings_dict, 
            dataset_name=clean_title, 
            p_value=p_value,
            save_path=save_file_path
        )


Processing Outliers for: Ori Curves
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/msc_outlier/ori_curves_msc_outlier_results.png

Processing Outliers for: Ori Curves Avg
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/msc_outlier/ori_curves_avg_msc_outlier_results.png

Processing Outliers for: Original Fitted Full
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/msc_outlier/original_fitted_full_msc_outlier_results.png

Processing Outliers for: Original Fitted Stretched
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/msc_outlier/original_fitted_stretched_msc_outlier_results.png

Processing Outliers for: Cleaned Std Fitted Full
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run 

# 3. AMF Outlier Detection

In [11]:
amf_features_all = ["Fm", "Fb", "Sc", "Cs", "Send", "Send_abs", "Send_fit", "Send_fit_abs"]
amf_plot_path = f"{exp_path}/amf_outlier/"

# Ensure the directory exists
os.makedirs(amf_plot_path, exist_ok=True)

if (not os.path.exists(save_path)):
    for name, features_df in zip(dataset_name, kinetic_features):
        
        clean_title = name.replace("_", " ").title()
        print(f"Generating plot for: {clean_title}...")
        
        # Construct the full file path for the boxplot
        file_name = f"{name}_amf_features_boxplot.png"
        save_file_path = os.path.join(amf_plot_path, file_name)
        
        feature_boxplot(
            features_df=features_df, 
            well_labels=Y_well, 
            feature_columns=amf_features_all, 
            title=f"Kinetic Features: {clean_title}",
            save_path=save_file_path
        )

Generating plot for: Ori Curves...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/amf_outlier/ori_curves_amf_features_boxplot.png
Generating plot for: Ori Curves Avg...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/amf_outlier/ori_curves_avg_amf_features_boxplot.png
Generating plot for: Original Fitted Full...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/amf_outlier/original_fitted_full_amf_features_boxplot.png
Generating plot for: Original Fitted Stretched...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/amf_outlier/original_fitted_stretched_amf_features_boxplot.png
Generating plot for: Cleaned Std Fitted Full...
  -> Saved: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KH

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

# ====================================================================
# MODULE 1: OUTLIER DETECTION
# ====================================================================

def detect_amf_outliers(features_df, Y_well, amf_features_basic, amf_features_sends):
    """
    Runs Isolation Forest anomaly detection for multiple features and appends 
    the labels directly to the passed features DataFrame.
    """
    unique_wells = np.unique(Y_well)
    
    for send in amf_features_sends:
        amf_features = amf_features_basic + [send]
        label_col = f"amf_label_{send}"
        
        # Initialize column
        features_df[label_col] = np.nan
        
        for well in unique_wells:
            well_mask = (Y_well == well)
            valid_mask = well_mask & ~features_df[amf_features].isna().any(axis=1)
            
            features_clean = features_df.loc[valid_mask, amf_features].values
            
            if len(features_clean) == 0:
                continue
                
            # Fit and predict with Isolation Forest
            clf = IsolationForest(random_state=0).fit(features_clean)
            features_df.loc[valid_mask, label_col] = clf.predict(features_clean)
            
    return features_df


# ====================================================================
# MODULE 2: VISUALIZATION (UPDATED TO 4 ROWS)
# ====================================================================

def plot_amf_outliers(features_df, curves_2d, reference_curves_2d, Y_well, amf_features_sends, dataset_name, file_prefix, save_dir=None):
    """
    Plots a transposed grid: 
    Row 1: Reference Curves - Normal (Original dataset[0])
    Row 2: Reference Curves - Outliers (Original dataset[0])
    Row 3: Normal Curves (Current dataset)
    Row 4: Outlier Curves (Current dataset)
    Columns: Wells (up to 10)
    Generates one figure per 'send' feature.
    """
    unique_wells = np.unique(Y_well)
    num_cols = min(len(unique_wells), 10) # Limit columns to 10
    
    for send in amf_features_sends:
        label_col = f"amf_label_{send}"
        
        # Expanded figure height for 4 rows
        fig = plt.figure(figsize=(num_cols * 4, 16))
        
        for i, well in enumerate(unique_wells):
            if i >= num_cols:
                break
                
            # Masks and Data Extraction
            well_mask = (Y_well == well)
            well_df = features_df[well_mask]
            
            is_outlier = (well_df[label_col] == -1).fillna(False).values
            
            # Current Dataset Curves
            well_curves = curves_2d[well_mask]
            normal_curves = well_curves[~is_outlier]
            outlier_curves = well_curves[is_outlier]
            mean_curr = np.nanmean(well_curves, axis=0) if len(well_curves) > 0 else None
            
            # Reference Dataset Curves (dataset[0])
            well_ref_curves = reference_curves_2d[well_mask]
            normal_ref = well_ref_curves[~is_outlier]
            outlier_ref = well_ref_curves[is_outlier]
            mean_ref = np.nanmean(well_ref_curves, axis=0) if len(well_ref_curves) > 0 else None
            
            
            # --- ROW 1: REFERENCE CURVES - NORMAL ---
            ax_ref_norm = fig.add_subplot(4, num_cols, i + 1)
            
            if len(normal_ref) > 0:
                ax_ref_norm.plot(normal_ref.T, c=f"C{i}", alpha=0.3)
            if mean_ref is not None:
                ax_ref_norm.plot(mean_ref, c="black", linewidth=2.5)
                
            ax_ref_norm.set_title(f"Well {well} (Ref Normal)\n(n={len(normal_ref)})", fontweight='bold')
            if i == 0:
                ax_ref_norm.set_ylabel("Original\nFluorescence")
                
                
            # --- ROW 2: REFERENCE CURVES - OUTLIERS ---
            ax_ref_outl = fig.add_subplot(4, num_cols, num_cols + i + 1, sharey=ax_ref_norm)
            
            if len(outlier_ref) > 0:
                ax_ref_outl.plot(outlier_ref.T, c="red", alpha=0.5, linewidth=1.5)
            if mean_ref is not None:
                ax_ref_outl.plot(mean_ref, c="black", linewidth=2.5)
                
            ax_ref_outl.set_title(f"Well {well} (Ref Outliers)\n(n={len(outlier_ref)})", fontweight='bold')
            if i == 0:
                ax_ref_outl.set_ylabel("Original\nFluorescence")


            # --- ROW 3: CURRENT CURVES - NORMAL ---
            ax_norm = fig.add_subplot(4, num_cols, 2 * num_cols + i + 1)
            
            if len(normal_curves) > 0:
                ax_norm.plot(normal_curves.T, c=f"C{i}", alpha=0.3)
            if mean_curr is not None:
                ax_norm.plot(mean_curr, c="black", linewidth=2.5)
                
            ax_norm.set_title(f"Normal Curves ({dataset_name})\n(n={len(normal_curves)})", fontweight='bold')
            if i == 0:
                ax_norm.set_ylabel("Processed\nFluorescence")
                
                
            # --- ROW 4: CURRENT CURVES - OUTLIERS ---
            ax_outl = fig.add_subplot(4, num_cols, 3 * num_cols + i + 1, sharey=ax_norm)
            
            if len(outlier_curves) > 0:
                ax_outl.plot(outlier_curves.T, c="red", alpha=0.5, linewidth=1.5)
            if mean_curr is not None:
                ax_outl.plot(mean_curr, c="black", linewidth=2.5)
                
            ax_outl.set_title(f"Outlier Curves ({dataset_name})\n(n={len(outlier_curves)})", fontweight='bold')
            ax_outl.set_xlabel("Time/Cycle")
            if i == 0:
                ax_outl.set_ylabel("Processed\nFluorescence")
                
        fig.suptitle(f"Isolation Forest Outliers | {dataset_name} | Feature: {send}", fontsize=18, fontweight='bold', y=1.02)
        plt.tight_layout()
        
        # --- CHANGED: Save and Close instead of Show ---
        if save_dir:
            save_path = os.path.join(save_dir, f"{file_prefix}_amf_outliers_{send}.png")
            plt.savefig(save_path, bbox_inches='tight', dpi=300, facecolor='white')
            print(f"  -> Saved plot to: {save_path}")
            
        plt.close(fig)

In [ ]:
# ====================================================================
# MODULE 3: MAIN EXECUTION LOOP
# ====================================================================

amf_features_basic = ["Fm", "Fb", "Sc", "Cs"]
amf_features_sends = ["Send", "Send_abs", "Send_fit", "Send_fit_abs"]

# Capture the baseline/reference dataset (dataset[0]) to pass into the plots
reference_dataset_curves = dataset[0]
amf_plot_path = f"{exp_path}/amf_outlier/"

# Ensure the target directory exists
os.makedirs(amf_plot_path, exist_ok=True)

if (not os.path.exists(save_path)):
    # Loop through all datasets (original, moving averages, fitted, stretched)
    for name, features_df, curves_2d in zip(dataset_name, kinetic_features, dataset):
        
        clean_title = name.replace("_", " ").title()
        print(f"\n{'='*60}\nProcessing AMF Outliers for: {clean_title}\n{'='*60}")
        
        # 1. Detect Outliers (Append columns in-place)
        features_df = detect_amf_outliers(
            features_df=features_df, 
            Y_well=Y_well, 
            amf_features_basic=amf_features_basic, 
            amf_features_sends=amf_features_sends
        )
        
        # 2. Plot Transposed 4-Row Grid and Save
        plot_amf_outliers(
            features_df=features_df, 
            curves_2d=curves_2d, 
            reference_curves_2d=reference_dataset_curves,
            Y_well=Y_well, 
            amf_features_sends=amf_features_sends, 
            dataset_name=clean_title,
            file_prefix=name,         # Pass the clean file string (e.g. 'ori_curves')
            save_dir=amf_plot_path    # Pass the destination directory
        )


Processing AMF Outliers for: Ori Curves
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/amf_outlier/ori_curves_amf_outliers_Send.png
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/amf_outlier/ori_curves_amf_outliers_Send_abs.png
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/amf_outlier/ori_curves_amf_outliers_Send_fit.png
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/amf_outlier/ori_curves_amf_outliers_Send_fit_abs.png

Processing AMF Outliers for: Ori Curves Avg
  -> Saved plot to: /Users/kautsarg/Documents/Final Project/Run Data/trial test data/D20250808_E00_C00_F4500KHz_U_Sample_7/amf_outlier/ori_curves_avg_amf_outliers_Send.png
  -> Saved plot to: /Users/kautsarg/Documents/Final Pr

# 4. Mean based outlier detection

In [ ]:
import numpy as np
import warnings
import matplotlib.pyplot as plt

# ====================================================================
# MODULE 1: OUTLIER DETECTION (MEAN ± STD) [NaN-SAFE]
# ====================================================================

def detect_mean_std_outliers(features_df, curves_2d, Y_well, num_std=3):
    """
    Detects outliers by checking if any point in a curve falls outside 
    the (mean ± num_std * std) envelope. Safely ignores NaN values.
    """
    unique_wells = np.unique(Y_well)
    col_name = f"mean_std_outlier_label_{num_std}sigma"
    
    # Initialize column
    features_df[col_name] = np.nan
    
    for well in unique_wells:
        well_mask = (Y_well == well)
        well_curves = curves_2d[well_mask]
        
        if len(well_curves) == 0:
            continue
            
        # Calculate dynamic mean and standard deviation (ignoring NaNs)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            cycle_data_mean = np.nanmean(well_curves, axis=0)
            cycle_data_std = np.nanstd(well_curves, axis=0)
        
        cycle_data_mean_upper = cycle_data_mean + (num_std * cycle_data_std)
        cycle_data_mean_lower = cycle_data_mean - (num_std * cycle_data_std)
        
        # Check if ANY point in the curve breaches the upper or lower bounds.
        # Note: NaN < Number safely evaluates to False in numpy.
        is_outside_lower = well_curves < cycle_data_mean_lower
        is_outside_upper = well_curves > cycle_data_mean_upper
        is_outside = is_outside_lower | is_outside_upper
        
        is_outlier = np.any(is_outside, axis=1)
        
        # Label: -1 for Outlier, 1 for Normal
        labels = np.where(is_outlier, -1, 1)
        features_df.loc[well_mask, col_name] = labels
        
    return features_df, col_name

# ====================================================================
# MODULE 2: VISUALIZATION (OPTIMIZED)
# ====================================================================

def plot_mean_std_outliers(features_df, curves_2d, reference_curves_2d, Y_well, col_name, dataset_name, num_std, save_path=None):
    unique_wells = np.unique(Y_well)
    num_cols = min(len(unique_wells), 10) 
    
    # OPTIMIZATION 1: Use constrained layout natively, much faster than tight_layout
    fig = plt.figure(figsize=(num_cols * 4, 16), layout="constrained")
    
    for i, well in enumerate(unique_wells):
        if i >= num_cols:
            break
            
        well_mask = (Y_well == well)
        well_df = features_df[well_mask]
        
        is_outlier = (well_df[col_name] == -1).fillna(False).values
        
        well_curr_curves = curves_2d[well_mask]
        normal_curr = well_curr_curves[~is_outlier]
        outlier_curr = well_curr_curves[is_outlier]
        
        well_ref_curves = reference_curves_2d[well_mask]
        normal_ref = well_ref_curves[~is_outlier]
        outlier_ref = well_ref_curves[is_outlier]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            mean_curr = np.nanmean(well_curr_curves, axis=0) if len(well_curr_curves) > 0 else None
            std_curr = np.nanstd(well_curr_curves, axis=0) if len(well_curr_curves) > 0 else None
            
            mean_ref = np.nanmean(well_ref_curves, axis=0) if len(well_ref_curves) > 0 else None
            std_ref = np.nanstd(well_ref_curves, axis=0) if len(well_ref_curves) > 0 else None
        
        
        # --- ROW 1: REFERENCE CURVES - NORMAL ---
        ax_ref_norm = fig.add_subplot(4, num_cols, i + 1)
        if len(normal_ref) > 0:
            # OPTIMIZATION 3: rasterized=True flattens dense lines to save RAM
            ax_ref_norm.plot(normal_ref.T, c=f"C{i}", alpha=0.3, rasterized=True)
        if mean_ref is not None:
            ax_ref_norm.plot(mean_ref, c="black", linewidth=2.5) 
            ax_ref_norm.plot(mean_ref + (num_std * std_ref), c="black", linestyle="--", linewidth=1) 
            ax_ref_norm.plot(mean_ref - (num_std * std_ref), c="black", linestyle="--", linewidth=1) 
            
        ax_ref_norm.set_title(f"Well {well} (Ref Normal)\n(n={len(normal_ref)})", fontweight='bold')
        if i == 0: ax_ref_norm.set_ylabel("Original\nFluorescence")
            
            
        # --- ROW 2: REFERENCE CURVES - OUTLIERS ---
        ax_ref_outl = fig.add_subplot(4, num_cols, num_cols + i + 1, sharey=ax_ref_norm)
        if len(outlier_ref) > 0:
            ax_ref_outl.plot(outlier_ref.T, c="red", alpha=0.5, linewidth=1.5, rasterized=True)
        if mean_ref is not None:
            ax_ref_outl.plot(mean_ref, c="black", linewidth=2.5)
            ax_ref_outl.plot(mean_ref + (num_std * std_ref), c="black", linestyle="--", linewidth=1)
            ax_ref_outl.plot(mean_ref - (num_std * std_ref), c="black", linestyle="--", linewidth=1)
            
        ax_ref_outl.set_title(f"Well {well} (Ref Outliers)\n(n={len(outlier_ref)})", fontweight='bold')
        if i == 0: ax_ref_outl.set_ylabel("Original\nFluorescence")


        # --- ROW 3: CURRENT CURVES - NORMAL ---
        ax_norm = fig.add_subplot(4, num_cols, 2 * num_cols + i + 1)
        if len(normal_curr) > 0:
            ax_norm.plot(normal_curr.T, c=f"C{i}", alpha=0.3, rasterized=True)
        if mean_curr is not None:
            ax_norm.plot(mean_curr, c="black", linewidth=2.5)
            ax_norm.plot(mean_curr + (num_std * std_curr), c="black", linestyle="--", linewidth=1)
            ax_norm.plot(mean_curr - (num_std * std_curr), c="black", linestyle="--", linewidth=1)
            
        ax_norm.set_title(f"Normal Curves ({dataset_name})\n(n={len(normal_curr)})", fontweight='bold')
        if i == 0: ax_norm.set_ylabel("Processed\nFluorescence")
            
            
        # --- ROW 4: CURRENT CURVES - OUTLIERS ---
        ax_outl = fig.add_subplot(4, num_cols, 3 * num_cols + i + 1, sharey=ax_norm)
        if len(outlier_curr) > 0:
            ax_outl.plot(outlier_curr.T, c="red", alpha=0.5, linewidth=1.5, rasterized=True)
        if mean_curr is not None:
            ax_outl.plot(mean_curr, c="black", linewidth=2.5)
            ax_outl.plot(mean_curr + (num_std * std_curr), c="black", linestyle="--", linewidth=1)
            ax_outl.plot(mean_curr - (num_std * std_curr), c="black", linestyle="--", linewidth=1)
            
        ax_outl.set_title(f"Outlier Curves ({dataset_name})\n(n={len(outlier_curr)})", fontweight='bold')
        ax_outl.set_xlabel("Time/Cycle")
        if i == 0: ax_outl.set_ylabel("Processed\nFluorescence")
            
    fig.suptitle(f"Mean Envelope Outliers (±{num_std} Std Dev) | {dataset_name}", fontsize=18, fontweight='bold', y=1.02)
    
    # Removed plt.tight_layout() here!
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300, facecolor='white')
        print(f"  -> Saved plot to: {save_path}")
        
    # OPTIMIZATION 2: Force complete teardown of the figure
    fig.clf()
    plt.close(fig)

In [ ]:
import gc
# ====================================================================
# MODULE 3: MAIN EXECUTION LOOP
# ====================================================================

num_std_threshold = 3 
reference_dataset_curves = dataset[0]
meanstd_plot_path = f"{exp_path}/meanstd_outlier/"
os.makedirs(meanstd_plot_path, exist_ok=True)

if (not os.path.exists(save_path)):
    for name, features_df, curves_2d in zip(dataset_name, kinetic_features, dataset):
        
        clean_title = name.replace("_", " ").title()
        print(f"\n{'='*60}\nProcessing Mean/Std Outliers for: {clean_title}\n{'='*60}")
        
        features_df, assigned_col_name = detect_mean_std_outliers(
            features_df=features_df, 
            curves_2d=curves_2d,
            Y_well=Y_well, 
            num_std=num_std_threshold
        )
        
        file_name = f"{name}_meanstd_outliers_{num_std_threshold}sigma.png"
        save_file_path = os.path.join(meanstd_plot_path, file_name)
        
        plot_mean_std_outliers(
            features_df=features_df, 
            curves_2d=curves_2d, 
            reference_curves_2d=reference_dataset_curves,
            Y_well=Y_well, 
            col_name=assigned_col_name,
            dataset_name=clean_title,
            num_std=num_std_threshold,
            save_path=save_file_path 
        )
        
        # OPTIMIZATION 2: Force Python to empty the garbage bin after every heavy loop
        gc.collect()

-----

# 5. Save Features Extracted Data

In [ ]:
if (not os.path.exists(save_path)):
    save_data = {
        "timestamps": timestamps,
        "Y_well": Y_well,
        "dataset_name": dataset_name,
        "dataset": dataset,
        "kinetic_features": kinetic_features,
    }

    with open(save_path, 'wb') as f:
        pickle.dump(save_data, f)
    print(f"  -> Saved training prepared data to {save_path}")

else:
    with open(save_path, 'rb') as f:
        training_data = pickle.load(f)

    timestamps = training_data["timestamps"]
    Y_well = training_data["Y_well"]
    dataset_name = training_data["dataset_name"]
    dataset = training_data["dataset"]
    kinetic_features = training_data["kinetic_features"]

# 6. Model Training

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

import tensorflow as tf
from scikeras.wrappers import KerasClassifier

# ====================================================================
# NEURAL NETWORK SETUP
# ====================================================================
class myWrapper(KerasClassifier):
    pass

def create_model(input_size, output_size, kernel_size_1=5, kernel_size_2=3): 
    inputs = tf.keras.layers.Input(shape=(input_size, 1))
    x = tf.keras.layers.Conv1D(16, kernel_size_1, activation='relu')(inputs)
    x = tf.keras.layers.Conv1D(8, kernel_size_2, activation='relu')(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(output_size, activation='softmax')(x)
    
    model = tf.keras.models.Model(inputs=inputs, outputs=x)
    model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

In [ ]:
# ====================================================================
# MODULE 1: MODEL EVALUATION FUNCTION (WITH REAL-TIME ACCURACY)
# ====================================================================
def evaluate_outlier_filters(X_curves, features_df, y_encoded, outlier_filters, dataset_name, mode_name):
    """
    Evaluates CNN, KNN, and LR using specific outlier filters.
    X_curves: The 2D array of curves to train on.
    features_df: The DataFrame containing the outlier labels.
    """
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=0)
    
    X_FFI_full = X_curves[:, [-1]]
    results_dict = {}

    for f in outlier_filters:
        filter_name = f if f else 'None (Baseline)'
        print(f"  -> Testing Filter: {filter_name}")
        
        # 1. Generate robust mask
        if f is None:
            mask = np.ones(len(y_encoded), dtype=bool)
        elif f in features_df.columns:
            mask = (features_df[f] == 1).fillna(False).values
        else:
            print(f"     [Warning] {f} not found in dataset. Skipping.")
            continue

        X_AC = X_curves[mask]
        X_FFI = X_FFI_full[mask]
        y_true = y_encoded[mask]

        if len(np.unique(y_true)) < 2:
            print(f"     [Warning] Not enough classes left after filtering. Skipping.")
            continue

        y_trues_, y_preds_AC_, y_preds_AC_kNN_, y_preds_FFI_ = [], [], [], []

        # 2. Train / Test Split
        splits = sss.split(X_AC, y_true)
        
        for train_index, test_index in splits:
            X_AC_train, X_AC_test = X_AC[train_index], X_AC[test_index]
            X_FFI_train, X_FFI_test = X_FFI[train_index], X_FFI[test_index]
            y_train, y_test = y_true[train_index], y_true[test_index]
            y_trues_.append(y_test)

            # --- Neural Network (AC) ---
            clf_AC = myWrapper(model=create_model,
                               model__input_size=X_AC.shape[1],
                               model__output_size=len(np.unique(y_encoded)),
                               epochs=1000, batch_size=512, shuffle=True, verbose=False)
            clf_AC.fit(X_AC_train, y_train)
            pred_AC = clf_AC.predict(X_AC_test)
            y_preds_AC_.append(pred_AC)
            
            cnn_acc = accuracy_score(y_test, pred_AC) * 100
            print(f"     [+] {mode_name}-{dataset_name}-{filter_name} | CNN (ACA) | {cnn_acc:5.2f}%")
            tf.keras.backend.clear_session()

            # --- K-Nearest Neighbors (AC) ---
            # REMOVED n_jobs=-1 to prevent BrokenProcessPool crashes
            clf_AC_kNN = KNeighborsClassifier(n_neighbors=10)
            clf_AC_kNN.fit(X_AC_train, y_train)
            pred_kNN = clf_AC_kNN.predict(X_AC_test)
            y_preds_AC_kNN_.append(pred_kNN)
            
            knn_acc = accuracy_score(y_test, pred_kNN) * 100
            print(f"     [+] {mode_name}-{dataset_name}-{filter_name} | KNN (ACA) | {knn_acc:5.2f}%")

            # --- Logistic Regression (FFI) ---
            # REMOVED n_jobs=-1 to prevent BrokenProcessPool crashes
            clf_FFI = LogisticRegression(max_iter=1000)
            clf_FFI.fit(X_FFI_train, y_train)
            pred_FFI = clf_FFI.predict(X_FFI_test)
            y_preds_FFI_.append(pred_FFI)
            
            lr_acc = accuracy_score(y_test, pred_FFI) * 100
            print(f"     [+] {mode_name}-{dataset_name}-{filter_name} | LR (FFI)  | {lr_acc:5.2f}%")
            
        # Store results for this filter
        results_dict[f] = {
            "y_trues_": y_trues_, "y_preds_AC_": y_preds_AC_,
            "y_preds_AC_kNN_": y_preds_AC_kNN_, "y_preds_FFI_": y_preds_FFI_,
            "mask_count": np.sum(mask)
        }

    return results_dict

In [ ]:
# ====================================================================
# MODULE 2: VISUALIZATION FUNCTIONS (SHOW & SAVE)
# ====================================================================
def plot_ml_results(results_dict, outlier_filters, dataset_name, mode_name, total_count, save_prefix=None):
    
    # Setup labels and colors
    filter_labels = [str(f) if f is not None else "No Filter" for f in outlier_filters if f in results_dict]
    
    colors = []
    for f in outlier_filters:
        if f not in results_dict: continue
        if f is None: colors.append('#888888')
        elif 'mean_' in f: colors.append('#ff7f0e')
        elif 'amf_' in f: colors.append('#2ca02c')
        else: colors.append('#9467bd')

    method_info = [
        ('Logistic Regression (FFI)', 'y_preds_FFI_'),
        ('kNN (ACA)', 'y_preds_AC_kNN_'),
        ('Convolutional Neural Network (ACA)', 'y_preds_AC_')
    ]

    # --- 1. PLOT ACCURACIES ---
    fig_acc, axes = plt.subplots(len(method_info), 1, figsize=(14, 18))
    
    for ax, (title, m_key) in zip(axes, method_info):
        means, stds = [], []
        base_mean = 0

        for f in outlier_filters:
            if f not in results_dict: continue
            res = results_dict[f]
            
            fold_accs = [accuracy_score(yt, yp) * 100 for yt, yp in zip(res['y_trues_'], res[m_key])]
            m_val = np.mean(fold_accs)
            s_val = np.std(fold_accs)
            
            means.append(m_val)
            stds.append(s_val)
            if f is None:
                base_mean = m_val

        bar_labels = [f'{m:.1f}%' for m in means]
        bars = ax.bar(filter_labels, means, yerr=stds, color=colors, edgecolor='black', alpha=0.8, capsize=5)
        
        ax.axhline(y=base_mean, color='red', linestyle='--', linewidth=2, label=f'Baseline ({base_mean:.1f}%)')
        ax.bar_label(bars, labels=bar_labels, padding=5, fontsize=10, fontweight='bold')
        
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_ylabel('Accuracy (%)')
        ax.set_ylim(0, 115) 
        ax.set_xticks(range(len(filter_labels)))
        ax.set_xticklabels(filter_labels, rotation=15, ha='right')
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        ax.legend(loc='upper right')

    fig_acc.suptitle(f"Model Accuracies | {mode_name}: {dataset_name}", fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    # Save first, then show
    if save_prefix:
        acc_path = f"{save_prefix}_accuracies.png"
        fig_acc.savefig(acc_path, bbox_inches='tight', dpi=300, facecolor='white')
        print(f"  -> Saved accuracy plot to: {acc_path}")
    plt.show()

    # --- 2. PLOT DATA COMPOSITION ---
    n_normals = [results_dict[f]['mask_count'] for f in outlier_filters if f in results_dict]
    n_outliers = [total_count - n for n in n_normals]

    fig_comp, ax_comp = plt.subplots(figsize=(12, 6))
    ax_comp.bar(filter_labels, n_normals, color=colors, edgecolor='black', alpha=0.8, label='Normal')
    ax_comp.bar(filter_labels, n_outliers, bottom=n_normals, color='#ffcccc', edgecolor='black', alpha=0.6, label='Outlier')

    for i in range(len(filter_labels)):
        if n_normals[i] > 0:
            ax_comp.text(i, n_normals[i]/2, f'{(n_normals[i]/total_count)*100:.1f}%', ha='center', color='white', fontweight='bold')
        if n_outliers[i] > 0:
            ax_comp.text(i, n_normals[i] + (n_outliers[i]/2), f'{(n_outliers[i]/total_count)*100:.1f}%', ha='center', color='darkred', fontweight='bold')

    ax_comp.set_title(f"Data Composition | {mode_name}: {dataset_name}", fontsize=14, fontweight='bold')
    ax_comp.set_ylabel("Number of Samples")
    ax_comp.set_xticks(range(len(filter_labels)))
    ax_comp.set_xticklabels(filter_labels, rotation=15, ha='right')
    plt.tight_layout()
    
    # Save first, then show
    if save_prefix:
        comp_path = f"{save_prefix}_composition.png"
        fig_comp.savefig(comp_path, bbox_inches='tight', dpi=300, facecolor='white')
        print(f"  -> Saved composition plot to: {comp_path}")
    plt.show()

    # --- 3. CONSOLE LEADERBOARD PRINT ---
    print(f"\n  🏆 Top Combinations for {mode_name}: {dataset_name}")
    print("  " + "-"*95)
    
    all_results = []
    for title, m_key in method_info:
        for f in outlier_filters:
            if f not in results_dict: continue
            res = results_dict[f]
            fold_accs = [accuracy_score(yt, yp) * 100 for yt, yp in zip(res['y_trues_'], res[m_key])]
            mean_acc = np.mean(fold_accs)
            filt_name = str(f) if f is not None else "Baseline (None)"
            
            all_results.append((mean_acc, dataset_name, mode_name, title, filt_name))
            
    all_results.sort(key=lambda x: x[0], reverse=True)
    
    for i, (acc, d_name, m_name, method, filt) in enumerate(all_results[:3]):
        print(f"  {i+1}. {acc:6.2f}% | Data: {d_name[:15]:<15} | Model: {method[:20]:<20} | Filter: {filt}")
    print("  " + "-"*95 + "\n")
    
    return all_results

In [ ]:
import os
import pickle
import gc

# ====================================================================
# MAIN EXECUTION LOOP WITH CHECKPOINTING
# ====================================================================

os.environ['MallocStackLogging'] = '0'

model_plot_path = f"{exp_path}/model_performance/"
# Ensure the directory exists
os.makedirs(model_plot_path, exist_ok=True)

# 1. Define the save path for the master dictionary
results_file_path = os.path.join(model_plot_path, "all_ml_results.pkl")

# 2. Load existing dictionary if it exists, otherwise create a new one
if os.path.exists(results_file_path):
    print(f"[*] Loading existing results from {results_file_path}...")
    with open(results_file_path, 'rb') as f:
        all_ml_results = pickle.load(f)
else:
    print("[*] Initializing new master results dictionary...")
    all_ml_results = {}


encoder = LabelEncoder()
y_full = encoder.fit_transform(Y_well)

# Specific filters requested
outlier_filters = [None, "msc_label_0.001", "amf_label_Send", "mean_std_outlier_label_3sigma"]

# Capture the baseline dataset
reference_dataset_curves = dataset[0]

# Calculate total datasets for progress tracking
total_datasets = len(dataset_name)

# Iterate through all datasets
for idx, (name, features_df, curves_2d) in enumerate(zip(dataset_name, kinetic_features, dataset)):
    clean_title = name.replace("_", " ").title()
    total_samples = len(y_full)
    
    # Initialize dictionary keys for this specific dataset if not present
    if clean_title not in all_ml_results:
        all_ml_results[clean_title] = {}
    
    progress_pct = (idx / total_datasets) * 100

    # -------------------------------------------------------------
    # PART A: Train on Current Dataset (Native Data + Native Filters)
    # -------------------------------------------------------------
    print(f"\n{'='*60}")
    print(f"[{progress_pct:.1f}%] [PART A] Training Native Data: {clean_title}")
    print(f"{'='*60}")
    
    # Check if Part A was already completed
    if "Native" in all_ml_results[clean_title]:
        print("  -> [CACHE HIT] Loading Native results from saved dictionary. Skipping training.")
        results_A = all_ml_results[clean_title]["Native"]
    else:
        results_A = evaluate_outlier_filters(
            X_curves=curves_2d, 
            features_df=features_df, 
            y_encoded=y_full, 
            outlier_filters=outlier_filters, 
            dataset_name=clean_title, 
            mode_name="Native Training"
        )
        # Save to dictionary and write to disk immediately
        all_ml_results[clean_title]["Native"] = results_A
        with open(results_file_path, 'wb') as f:
            pickle.dump(all_ml_results, f)
            print("  -> [SAVED] Native results appended to dictionary on disk.")
    
    # Prefix for saving Part A plots
    prefix_A = os.path.join(model_plot_path, f"{name}_Native")
    plot_ml_results(results_A, outlier_filters, clean_title, "Native Training", total_samples, save_prefix=prefix_A)


    # -------------------------------------------------------------
    # PART B: Train on Reference Dataset (Raw Data + Native Filters)
    # -------------------------------------------------------------
    print(f"\n{'='*60}")
    print(f"[{progress_pct:.1f}%] [PART B] Training Reference Data using masks from: {clean_title}")
    print(f"{'='*60}")
    
    # Check if Part B was already completed
    if "Reference" in all_ml_results[clean_title]:
        print("  -> [CACHE HIT] Loading Reference results from saved dictionary. Skipping training.")
        results_B = all_ml_results[clean_title]["Reference"]
    else:
        results_B = evaluate_outlier_filters(
            X_curves=reference_dataset_curves,
            features_df=features_df, 
            y_encoded=y_full, 
            outlier_filters=outlier_filters, 
            dataset_name=clean_title, 
            mode_name="Reference Training"
        )
        # Save to dictionary and write to disk immediately
        all_ml_results[clean_title]["Reference"] = results_B
        with open(results_file_path, 'wb') as f:
            pickle.dump(all_ml_results, f)
            print("  -> [SAVED] Reference results appended to dictionary on disk.")
    
    # Prefix for saving Part B plots
    prefix_B = os.path.join(model_plot_path, f"{name}_Reference")
    plot_ml_results(results_B, outlier_filters, clean_title, "Reference Training", total_samples, save_prefix=prefix_B)

    # Force memory cleanup after each dataset cycle
    gc.collect()

# Final completion message
print(f"\n{'='*60}")
print(f"[100.0%] All {total_datasets} datasets processed successfully!")
print(f"Master dictionary saved at: {results_file_path}")
print(f"{'='*60}")